In [ ]:
%%capture
import warnings
warnings.filterwarnings("ignore")

from IPython.display import HTML, Markdown, display

import altair as alt
import calitp_portfolio.magics
import geopandas as gpd
import gcsfs
import google.auth
import pandas as pd

import _operator_report_utils as utils
from update_vars import (
    DIGEST_DICT, PROCESSED_GCS 
)

credentials, _ = google.auth.default()

In [ ]:
def formatted(number):
    return "{:,}".format(number)

In [ ]:
# parameters cell
#analysis_name = "Alameda-Contra Costa Transit District"

In [ ]:
%%capture_parameters
analysis_name

In [ ]:
# Set drop down menu to be on the upper right for the charts
display(
    HTML(
        """
<style>
form.vega-bindings {
  position: absolute;
  right: 0px;
  top: 0px;
}
</style>
"""
    )
)

In [ ]:
fct_monthly_routes_url = f"{PROCESSED_GCS}{DIGEST_DICT.route_map}.parquet"

fct_monthly_route_df = gpd.read_parquet(
    fct_monthly_routes_url,
    filters=[[("Analysis Name", "==", analysis_name)]],
    storage_options={"token": credentials.token}
).reset_index(drop=True)

In [ ]:
schedule_rt_route_direction_summary_url = f"{PROCESSED_GCS}{DIGEST_DICT.schedule_rt_route_direction}.parquet"

schedule_rt_route_direction_summary_df = pd.read_parquet(
    schedule_rt_route_direction_summary_url,
    filesystem = gcsfs.GCSFileSystem(),
    filters=[[("Analysis Name", "==", analysis_name)]]
).reset_index(drop=True)

In [ ]:
one_month_schedule_rt_route_direction_summary_df = (
    schedule_rt_route_direction_summary_df.loc[schedule_rt_route_direction_summary_df["Day Type"] == "Weekday"]
    .sort_values(by = ["Date", "Route"], ascending = [False, True])
    .drop_duplicates(subset = ["Route"])
    .reset_index(drop=True)
 )

In [ ]:
operator_hourly_summary_url = f"{PROCESSED_GCS}{DIGEST_DICT.hourly_day_type_summary}.parquet"

operator_hourly_summary_df = pd.read_parquet(
    operator_hourly_summary_url,
    filesystem = gcsfs.GCSFileSystem(),
    filters=[[("Analysis Name", "==", analysis_name),
    ("Departure Hour", "<=", 24)]]
).reset_index(drop=True)

In [ ]:
ntd_url = f"{PROCESSED_GCS}{DIGEST_DICT.ntd_profile}.parquet"

ntd_profile_df = pd.read_parquet(
    ntd_url, 
    filesystem = gcsfs.GCSFileSystem(),
    filters=[[("analysis_name", "==", analysis_name)]]
).reset_index(drop=True)

In [ ]:
operator_summary_url = f"{PROCESSED_GCS}{DIGEST_DICT.operator_summary}.parquet"

operator_df = pd.read_parquet(
    operator_summary_url,
    filesystem = gcsfs.GCSFileSystem(),
    filters=[
        ("Day Type", "==", "Weekday"),
        ("Analysis Name", "==", analysis_name)],
).drop_duplicates(
    subset = ["Analysis Name", "Date"]
).reset_index(drop=True)

# {analysis_name}

## Operator Overview

In [ ]:
try:
    service_area = formatted(ntd_profile_df.service_area_sq_miles.values[0])
    service_pop = formatted(ntd_profile_df.service_area_pop.values[0])
except:
    pass

In [ ]:
try:
    display(
        Markdown(
            f"""{analysis_name} is headquartered in <b>{ntd_profile_df.county_name.values[0]}</b> 
            County in the Urbanized Area of <b>{ntd_profile_df.primary_uza_name.values[0]}</b>.<br>
            This operator provides <b>{service_area}</b> square miles of public transit service, 
            which has a service population of <b>{service_pop}</b>.<br>
            This organization is a {ntd_profile_df.reporter_type.values[0]}.<br>
            <b>Data Source</b>: 
            <a href="https://www.transit.dot.gov/ntd/data-product/2022-annual-database-agency-information">National Transit Database</a> 
            Annual Agency Information.
            """
        )
    )
except:
    pass

## Route Typologies

In [ ]:
try:
    n_routes = one_month_schedule_rt_route_direction_summary_df["Route"].nunique()
    display(
        Markdown(
            f"""{analysis_name} runs <b>{n_routes}</b> unique routes. Below is the breakdown of the routes.<p>
            Route categories are determined by a combination of GTFS `route_type` 
            and `route_short_name` and `route_long_name`. 
            Please see the 
            <a href="https://github.com/cal-itp/gtfs-curator/blob/main/gtfs_digest/methodology.md">methodology docs</a>
            for more details on this approach.
        """
        )
    )
except:
    display(Markdown(f"""{analysis_name} doesn't have an operator profile."""))

In [ ]:
try:
    display(utils.create_route_typology(one_month_schedule_rt_route_direction_summary_df))
except:
    display(Markdown(f"""{analysis_name} doesn't have information on route typologies."""))

## Route Length

In [ ]:
try:
    display(utils.create_route_lengths(fct_monthly_route_df))
except:
    display(Markdown(f"""{analysis_name} doesn't have information on route lengths."""))

## Service Area

In [ ]:
fct_monthly_route_df2 = fct_monthly_route_df[["Route Name", "Geometry"]].drop_duplicates().reset_index(drop=True)

if len(fct_monthly_route_df2) > 0:
    m = fct_monthly_route_df2.explore(
        "Route Name",
        name = f"Routes for {analysis_name}",
        categorical = True,
        cmap = "Spectral",
        legend = False,
        tiles = "CartoDB Positron",
        #marker_kwds={"fill": True},
        #style_kwds={"opacity": 0.5, "fillOpacity": 0.3}
    )
    display(m)
else:
    display(Markdown(f"""{analysis_name} doesn't have an route geographies."""))


## Service Hours 

In [ ]:
try:
    display(utils.create_hourly_summary(operator_hourly_summary_df))
except: 
    display(Markdown(f"""{analysis_name} doesn't have service hours information."""))

## Trip Update Overview

In [ ]:
tu_cols = ["TU Messages Per Minute", "Pct TU Trips"]

if len(operator_df.dropna(subset=tu_cols)) > 0:
    display(utils.create_tu_minute(operator_df))
    display(utils.create_tu_pct(operator_df))
else:
    display(Markdown(f"""{analysis_name} doesn't have trip update information."""))

## Vehicle Positions Overview

In [ ]:
vp_cols = ["VP Messages Per Minute", "Pct VP Trips"]

if len(operator_df.dropna(subset=vp_cols)) > 0:

    display(utils.create_vp_minute(operator_df))
    display(utils.create_vp_pct(operator_df))
else:
    display(Markdown(f"""{analysis_name} doesn't have vehicle positions information."""))

## Detailed Route Overview

In [ ]:
alt.data_transformers.enable("vegafusion")
utils.create_route_dropdown_chart(schedule_rt_route_direction_summary_df)